<a href="https://colab.research.google.com/github/heisenberg304/aprendiendo-ml-cloud/blob/semana-6/Dia%204.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Cargar librerias
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, f1_score
from sklearn.model_selection import cross_val_score

#Cargar dataset
data = pd.read_csv("/content/drive/MyDrive/data_csv/heart_disease_uci.csv")

#===================================================================================================================================================================================================

#MANIPULAR CSV
  #rellenar: trestbps(59)(median o mean), chol(30)(median), fbs(90)(mode), restecg(2)(mode), thalch(55)(median o mean), oldpeak(62)(median)
data["trestbps"].fillna(data["trestbps"].median(), inplace=True)
data["chol"].fillna(data["chol"].median(), inplace=True)
data["fbs"].fillna(data["fbs"].mode()[0], inplace=True)
data["restecg"].fillna(data["restecg"].mode()[0], inplace=True)
data["thalch"].fillna(data["thalch"].median(), inplace=True)
data["oldpeak"].fillna(data["oldpeak"].median(), inplace=True)

  #target binario
data["num"] = data["num"].apply(lambda x: 1 if x > 0 else 0)

  #encoding: sex, cp, restecg, num (target)
fe_ma = pd.get_dummies(data["sex"])
CP = pd.get_dummies(data["cp"])
RESTECG = pd.get_dummies(data["restecg"])

  #union de datasets
data = pd.concat([data, fe_ma, CP, RESTECG], axis=1)
  #borrar columnas
data = data.drop(["id", "dataset", "slope", "ca", "thal", "sex", "cp", "restecg"], axis=1)

#===================================================================================================================================================================================================

#crear x and y
x = data[['age','trestbps','chol','fbs','thalch','exang',
          'oldpeak','Female','Male','asymptomatic',
          'atypical angina','non-anginal','typical angina',
          'lv hypertrophy','normal','st-t abnormality']]
y = data["num"]

#split 80/20
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size = 0.2,
    random_state=55
)

#===================================================================================================================================================================================================

#crear modelo
model = RandomForestClassifier(n_estimators=200,
                               max_depth=5,
                               min_samples_leaf=17)
#entrenar modelo
model.fit(x_train, y_train)
#prediccion
#pred = model.predict(x_test)
#threshold
probs = model.predict_proba(x_test)[:, 1]
threshold = 0.35
pred = (probs >= threshold).astype(int)
#Importancia de cada feature
importances = model.feature_importances_

#train_score, usar train en vez de test en predict
probs_train = model.predict_proba(x_train)[:, 1]
threshold = 0.35
pred_train = (probs_train >= threshold).astype(int)

#validacion cruzada
scores = cross_val_score(
    model,
    x,
    y,
    cv=5
)

#metricas
matrix = confusion_matrix(y_test, pred)
accuracy = accuracy_score(y_test, pred)
train_score = accuracy_score(y_train, pred_train)
precision = precision_score(y_test, pred)
recall = recall_score(y_test, pred)
f1 = f1_score(y_test, pred)
#print de metricas
print(f"matrix -> {matrix}")
print(f"accuracy -> {accuracy}")
print(f"precision -> {precision}")
print(f"recall -> {recall}")
print(scores)
print(f"mean -> {scores.mean()}")
print(f"std -> {scores.std()}")
print(f"train score -> {train_score}")
print(f"gap -> {train_score - scores.mean()}")


/tmp/ipykernel_11216/2354369436.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data["trestbps"].fillna(data["trestbps"].median(), inplace=True)
/tmp/ipykernel_11216/2354369436.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, 

matrix -> [[52 32]
 [ 6 94]]
accuracy -> 0.7934782608695652
precision -> 0.746031746031746
recall -> 0.94
[0.76630435 0.72826087 0.85869565 0.85326087 0.66304348]
mean -> 0.7739130434782608
std -> 0.07472381409694442
train score -> 0.8002717391304348
gap -> 0.02635869565217397


In [ ]:

#cargar librerias
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree, export_graphviz
from sklearn.model_selection import cross_val_score


#Cargar dataset
data = pd.read_csv("/content/drive/MyDrive/data_csv/heart_disease_uci.csv")


#manipular dataset
#rellenar: trestbps(59)(median o mean), chol(30)(median), fbs(90)(mode), restecg(2)(mode), thalch(55)(median o mean), oldpeak(62)(median)
data["trestbps"].fillna(data["trestbps"].median(), inplace=True)
data["chol"].fillna(data["chol"].median(), inplace=True)
data["fbs"].fillna(data["fbs"].mode()[0], inplace=True)
data["restecg"].fillna(data["restecg"].mode()[0], inplace=True)
data["thalch"].fillna(data["thalch"].median(), inplace=True)
data["oldpeak"].fillna(data["oldpeak"].median(), inplace=True)

#target binario
data["num"] = data["num"].apply(lambda x: 1 if x > 0 else 0)

#encoding: sex, cp, restecg, num (target)
fe_ma = pd.get_dummies(data["sex"])
CP = pd.get_dummies(data["cp"])
RESTECG = pd.get_dummies(data["restecg"])

#union de datasets
data = pd.concat([data, fe_ma, CP, RESTECG], axis=1)
#borrar columnas
data = data.drop(["id", "dataset", "slope", "ca", "thal", "sex", "cp", "restecg"], axis=1)


#crear x,y
x = data[['age','trestbps','chol','fbs','thalch','exang',
          'oldpeak','Female','Male','asymptomatic',
          'atypical angina','non-anginal','typical angina',
          'lv hypertrophy','normal','st-t abnormality']]
y = data["num"]
#split 80/20
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size = 0.2,
    random_state=55
)

#crear modelo
model = DecisionTreeClassifier(max_depth=6, min_samples_leaf=14)
#entrenar modelo
model.fit(x_train, y_train)
#prediccion
#pred = model.predict(x_test)
#threshold
probs = model.predict_proba(x_test)[:, 1]
threshold = 0.4
pred = (probs >= threshold).astype(int)

#Importancia de cada feature
importances = model.feature_importances_
#profundidad del Tree
depth = model.tree_.max_depth


#train_score, usar train en vez de test en predict
probs_train = model.predict_proba(x_train)[:, 1]
threshold = 0.4
pred_train = (probs_train >= threshold).astype(int)

#validacion cruzada
scores = cross_val_score(
    model,
    x,
    y,
    cv=5
)

#metricas
matrix = confusion_matrix(y_test, pred)
accuracy = accuracy_score(y_test, pred)
train_score = accuracy_score(y_train, pred_train)
precision = precision_score(y_test, pred)
recall = recall_score(y_test, pred)
f1 = f1_score(y_test, pred)
#print de metricas
print(f"matrix -> {matrix}")
print(f"accuracy -> {accuracy}")
print(f"precision -> {precision}")
print(f"recall -> {recall}")
print(scores)
print(f"mean -> {scores.mean()}")
print(f"std -> {scores.std()}")
print(f"train score -> {train_score}")
print(f"gap -> {train_score - scores.mean()}")


matrix -> [[49 35]
 [13 87]]
accuracy -> 0.7391304347826086
precision -> 0.7131147540983607
recall -> 0.87
[0.69565217 0.70108696 0.88586957 0.82608696 0.64673913]
mean -> 0.7510869565217392
std -> 0.08972495253690846
train score -> 0.8192934782608695
gap -> 0.06820652173913033


/tmp/ipykernel_3775/1651998396.py:31: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data["trestbps"].fillna(data["trestbps"].median(), inplace=True)
/tmp/ipykernel_3775/1651998396.py:32: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, in